In [18]:
import pandas as pd
import random
import os
import io
import time
import requests
from datetime import datetime
import sportsdataverse.nfl as sdv
from collections import deque

class RateLimiter:
    def __init__(self, max_requests=20, time_window=60):  # 20 per minute
        self.max_requests = max_requests
        self.time_window = time_window
        self.requests = deque()
    
    def wait_if_needed(self):
        now = time.time()
        # Remove requests older than 1 minute
        while self.requests and now - self.requests[0] >= self.time_window:
            self.requests.popleft()
        
        # If at limit, wait until oldest request is 1 minute old
        if len(self.requests) >= self.max_requests:
            sleep_time = self.time_window - (now - self.requests[0]) + 1
            print(f"Rate limit reached. Waiting {sleep_time:.1f} seconds...")
            time.sleep(sleep_time)
        
        self.requests.append(now)

rate_limiter = RateLimiter(20, 60)

def polite_fetch_table(url):
    rate_limiter.wait_if_needed()
    tables = pd.read_html(url)
    return tables

def import_games(start_year, end_year=None):
    years = range(start_year, end_year + 1) if end_year else [start_year]
    all_games = pd.concat([sdv.espn_nfl_schedule(dates=year, return_as_pandas=True) for year in years], axis=0, ignore_index=True)
    rate_limiter.wait_if_needed()
    return all_games

def get_game_stats(team, start_year=int(datetime.now().year), end_year=None):
    team = team.lower()
    years = range(start_year, end_year + 1) if end_year else [start_year]
    team_df = pd.DataFrame()
    
    for year in years:
        url = f'https://www.pro-football-reference.com/teams/{team}/{year}.htm'
        try:
            df = polite_fetch_table(url)[1]
            team_df = pd.concat([team_df, df], axis=0, ignore_index=True)
        except Exception as e:
            print(f"Error retrieving data for {team} in {year}: {e}")
    
    return team_df

def get_data(dir_name):
    files = os.listdir(f'../../data/{dir_name}')
    curr_year = datetime.now().year
    
    if dir_name == 'team_jsons':
        os.makedirs('../../data/test_jsons', exist_ok=True)
        
        for idx, file in enumerate(files):
            file_edited = file.replace('.json', '')
            last_updated_year, team = int(file_edited.split('_')[-1]), file_edited.split('_')[0]
            
            print(f"Processing team {team} from year {last_updated_year + 1} to {curr_year}")
            
            new_team_data = get_game_stats(team, last_updated_year + 1, curr_year)
            
            if new_team_data is not None and not new_team_data.empty:
                data_df = pd.read_json(f'../../data/{dir_name}/{file}')
                
                data_df = pd.concat([data_df, new_team_data], axis=0, ignore_index=True)
                
                data_df.to_json(f'../../data/test_jsons/{team}_2002_{curr_year}.json', orient='records')
                print(f"Updated {team} data through {curr_year}")
            else:
                print(f"No new data available for {team}")
    else:
        os.makedirs('../../data/nfl_games', exist_ok=True)
        
        if not files:
            print("No files found in directory")
            return
            
        files_edited = files[0].replace('.json', '')
        last_updated_year = int(files_edited.split('_')[-1])
        
        print(f"Processing NFL games from year {last_updated_year} to {curr_year}")
        
        new_data = import_games(last_updated_year, curr_year)
        
        if new_data is not None and not new_data.empty:
            data_df = pd.read_json(f'../../data/{dir_name}/nfl_2002_{last_updated_year}.json')
            data_df = pd.concat([data_df, new_data], axis=0, ignore_index=True)
            data_df.to_json(f'../../data/nfl_games/nfl_2002_{curr_year}.json', orient='records')
            print(f"Updated NFL games through {curr_year}")
        else:
            print("No new NFL game data available")

def get_upcoming_data():
    print("Starting data update process...")
    print("Rate limit: 20 requests per minute")
    
    get_data('nfl_games')
    get_data('team_jsons')
    
    print("Data update process completed!")
    return

if __name__ == "__main__":
    get_upcoming_data()

Starting data update process...
Rate limit: 20 requests per minute
Processing NFL games from year 2023 to 2025
Updated NFL games through 2025
Processing team ARI from year 2025 to 2025
Error retrieving data for ari in 2025: HTTP Error 404: Not Found
No new data available for ARI
Processing team ATL from year 2025 to 2025
Error retrieving data for atl in 2025: list index out of range
No new data available for ATL
Processing team BAL from year 2025 to 2025
Error retrieving data for bal in 2025: HTTP Error 404: Not Found
No new data available for BAL
Processing team BUF from year 2025 to 2025
Error retrieving data for buf in 2025: list index out of range
No new data available for BUF
Processing team CAR from year 2025 to 2025
Error retrieving data for car in 2025: list index out of range
No new data available for CAR
Processing team CHI from year 2025 to 2025
Error retrieving data for chi in 2025: list index out of range
No new data available for CHI
Processing team CIN from year 2025 to 

In [10]:
old_df = pd.read_json(f'../../data/team_jsons/CHI_2002_2024.json')
main_df = pd.read_json(f'../../data/test_jsons/CHI_2002_2025.json')
main_df

FileNotFoundError: File ../../data/test_jsons/CHI_2002_2025.json does not exist

In [4]:
old_df

,"('Unnamed: 0_level_0', 'Week')","('Unnamed: 1_level_0', 'Day')","('Unnamed: 2_level_0', 'Date')","('Unnamed: 3_level_0', 'Unnamed: 3_level_1')","('Unnamed: 4_level_0', 'Unnamed: 4_level_1')","('Unnamed: 5_level_0', 'Unnamed: 5_level_1')","('Unnamed: 6_level_0', 'OT')","('Unnamed: 7_level_0', 'Rec')","('Unnamed: 8_level_0', 'Unnamed: 8_level_1')","('Unnamed: 9_level_0', 'Opp')",...,"('Offense', 'RushY')","('Offense', 'TO')","('Defense', '1stD')","('Defense', 'TotYd')","('Defense', 'PassY')","('Defense', 'RushY')","('Defense', 'TO')","('Expected Points', 'Offense')","('Expected Points', 'Defense')","('Expected Points', 'Sp. Tms')"
0,1,Sun,September 8,1:04PM ET,boxscore,W,None,1-0,None,Minnesota Vikings,...,80.0,2.0,19.0,368.0,228.0,140.0,3.0,6.37,-1.20,1.01
1,2,Sun,September 15,1:04PM ET,boxscore,W,None,2-0,@,Atlanta Falcons,...,106.0,2.0,19.0,257.0,135.0,122.0,3.0,-9.67,8.39,3.54
2,3,Sun,September 22,1:05PM ET,boxscore,L,None,2-1,None,New Orleans Saints,...,125.0,2.0,18.0,302.0,229.0,73.0,3.0,0.58,-0.64,-6.03
3,4,Sun,September 29,1:04PM ET,boxscore,L,OT,2-2,@,Buffalo Bills,...,52.0,NaN,26.0,410.0,307.0,103.0,1.0,3.14,-7.40,0.06
4,5,Mon,October 7,9:08PM ET,boxscore,L,None,2-3,None,Green Bay Packers,...,45.0,4.0,20.0,457.0,333.0,124.0,1.0,-12.87,-3.38,8.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,14,Sun,December 8,4:25PM ET,boxscore,L,None,4-9,@,San Francisco 49ers,...,68.0,1.0,22.0,452.0,321.0,131.0,1.0,-8.25,-26.53,9.47
404,15,Mon,December 16,8:00PM ET,boxscore,L,None,4-10,@,Minnesota Vikings,...,113.0,1.0,24.0,329.0,215.0,114.0,1.0,-14.01,-8.32,3.74
405,16,Sun,December 22,1:00PM ET,boxscore,L,None,4-11,None,Detroit Lions,...,59.0,2.0,27.0,475.0,329.0,146.0,NaN,3.86,-24.06,0.82
406,17,Thu,December 26,8:15PM ET,boxscore,L,None,4-12,None,Seattle Seahawks,...,103.0,1.0,12.0,265.0,143.0,122.0,1.0,-12.22,6.51,2.22


In [13]:
!pip install fake_useragent